In [1]:
using QuantumCollocation
using NamedTrajectories
using TrajectoryIndexingUtils

using CairoMakie
using DelimitedFiles
using Distributions
using LinearAlgebra
using SparseArrays
pathof(QuantumCollocation)

"/home/andy/Documents/Research/JuliaPackages/StealthPICO/src/QuantumCollocation.jl"

In [2]:
# Operators 
const n_levels = 2
const a⁺ = create(n_levels)
const a⁻ = annihilate(n_levels)

const H_ops = Dict(
        "X" => a⁻ + a⁺,
        "Y" => -im * (a⁻ - a⁺),
        "Z" => I - 2 * a⁺ * a⁻,
)

# Time
T = 45
Δt = 2/9

# Basic gates
I_gate = Matrix{ComplexF64}(I, 2, 2)
X_gate = GATES[:X]
SX_gate = sqrt(GATES[:X])

2×2 Matrix{ComplexF64}:
 0.5+0.5im  0.5-0.5im
 0.5-0.5im  0.5+0.5im

In [3]:
case = "6qubits"
if case == "4qubits"
    neighbors = [join([(i == j || i == mod(j, 4) + 1) ? "Z" : "I" for i in 1:4]) for j in 1:4]
    dephasing = [join([(i == j) ? "Z" : "I" for i in 1:4]) for j in 1:4]
    crosstalk_labels = vcat(neighbors, dephasing)

    H_controls_names = [join([i == j ? "X" : "I" for i in 1:4]) for j in 1:4]

    U_goal = reduce(kron, [X_gate, X_gate, SX_gate, SX_gate])
elseif case == "6qubits"
    neighbors = [join([(i == j || i == mod(j, 6) + 1) ? "Z" : "I" for i in 1:6]) for j in 1:6]
    diagonals = [join([(i == j || i == mod(j, 6) + 3) ? "Z" : "I" for i in 1:6]) for j in 1:3]
    # Dephasing unnecessary because of identity
    # dephasing = [join([(i == j) ? "Z" : "I" for i in 1:6]) for j in 1:6]
    crosstalk_labels = vcat(neighbors, diagonals)
    H_crosstalk = sum(kron_from_dict(n, H_ops) for n in crosstalk_labels)

    H_controls_names = [join([i == j ? "X" : "I" for i in 1:6]) for j in 1:6]
    
    U_goal = reduce(kron, [X_gate, X_gate, SX_gate, SX_gate, I_gate, I_gate])
end;

In [4]:
H_controls = [kron_from_dict(n, H_ops) for n in H_controls_names]
H_drift = zeros(n_levels^length(H_controls), n_levels^length(H_controls))
H_crosstalk = sum(kron_from_dict(n, H_ops) for n in crosstalk_labels)
;

In [5]:
@views function infidelity_robustness(Hₑ::AbstractMatrix, p::QuantumControlProblem)
    Z⃗ = vec(p.trajectory.data)
    Z = p.trajectory
    return InfidelityRobustnessObjective(Hₑ, Z).L(Z⃗, Z)
end

infidelity_robustness (generic function with 1 method)

# Original control problem

In [8]:
ops = Options()
ops.mumps_mem_percent = 5

prob = UnitarySmoothPulseProblem(
    H_drift,
    H_controls,
    U_goal,
    T,
    Δt;
    a_guess=(π/2/T/Δt) * ones((length(H_controls), T)),
    timesteps_all_equal=true,
    free_time=false,
    hessian_approximation=true,
    pade_order=20,
    R=10.,
    R_dda=5.,
    ipopt_options=ops
)

            memory estimate for stored jacobians: T * sizeof(∂f) = 45 * 0.026346336 = 1.18558512 GB


applying constraint: initial value of Ũ⃗
applying constraint: initial value of a
applying constraint: final value of a
applying constraint: bounds on a
applying constraint: bounds on dda


QuantumControlProblem(Ipopt.Optimizer, MathOptInterface.VariableIndex[MOI.VariableIndex(1) MOI.VariableIndex(8211) … MOI.VariableIndex(353031) MOI.VariableIndex(361241); MOI.VariableIndex(2) MOI.VariableIndex(8212) … MOI.VariableIndex(353032) MOI.VariableIndex(361242); … ; MOI.VariableIndex(8209) MOI.VariableIndex(16419) … MOI.VariableIndex(361239) MOI.VariableIndex(369449); MOI.VariableIndex(8210) MOI.VariableIndex(16420) … MOI.VariableIndex(361240) MOI.VariableIndex(369450)], QuantumSystem{Float64}(sparse(Int64[], Int64[], Float64[], 64, 64), sparse(Int64[], Int64[], Float64[], 64, 64), SparseMatrixCSC{Float64, Int64}[sparse([33, 34, 35, 36, 37, 38, 39, 40, 41, 42  …  23, 24, 25, 26, 27, 28, 29, 30, 31, 32], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  55, 56, 57, 58, 59, 60, 61, 62, 63, 64], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 64, 64), sparse([17, 18, 19, 20, 21, 22, 23, 24, 25, 26  …  39, 40, 41, 42, 43, 44, 45, 46, 47, 48]

In [9]:
solve!(prob; max_iter=40)

println("Fidelity: ", unitary_fidelity(prob))
println("Crosstalk robustness: ", infidelity_robustness(H_crosstalk, prob))



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.13, running with linear solver MUMPS 5.6.1.

Number of nonzeros in equality constraint Jacobian...: 47728164
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

MUMPS returned INFO(1) = -9 and requires more memory, reallocating.  Attempt 1
  Increasing icntl[13] from 5 to 10.
MUMPS returned INFO(1) = -9 and requires more memory, reallocating.  Attempt 2
  Increasing icntl[13] from 10 to 20.
MUMPS returned INFO(1) = -9 and requires more memory, reallocating.  Attempt 3
  Increasing icntl[13] from 20 to 40

   1  1.0235808e+02 4.76e-03 5.36e-01 -11.0 3.79e-01    -  1.00e+00 1.00e+00h  1


   2  1.0091423e+02 3.16e-03 1.52e+00 -11.0 3.33e-01    -  1.00e+00 1.00e+00h  1


In [ ]:
function plot_control(
    ts::AbstractVector, 
    as::Union{Nothing, AbstractMatrix}=nothing,
    das::Union{Nothing, AbstractMatrix}=nothing,
    ddas::Union{Nothing, AbstractMatrix}=nothing,
)
    # plot(prob.trajectory; comps=[:a, :da, :dda])
    f = Figure(resolution=(800, 600))
    colors = [:red, :green, :blue, :orange, :purple, :yellow, :black]

    if !isnothing(as)
        ax1 = Axis(f[1, 1])
        for i in 1:size(as, 1)
            lines!(ax1, ts, as[i, :]; color=colors[i])
        end
    end
    if !isnothing(das)
        ax2 = Axis(f[2, 1])
        for i in 1:size(as, 1)
            lines!(ax2, ts, das[i, :]; color=colors[i])
        end
    end
    if !isnothing(ddas)
        ax3 = Axis(f[3, 1])
        for i in 1:size(as, 1)
            lines!(ax3, ts, ddas[i, :]; color=colors[i])
        end
    end
    return f
end

function plot_control(prob::QuantumControlProblem)
    ts = accumulate(+, timesteps(prob.trajectory))
    as = prob.trajectory[:a]
    das = prob.trajectory[:da]
    ddas = prob.trajectory[:dda]
    return plot_control(ts, as, das, ddas)
    f = Figure(resolution=(800, 600))
end



In [ ]:
plot_control(prob)

In [ ]:
println("max(a): ", maximum(prob.trajectory[:a]))
println("max(dda): ", maximum(prob.trajectory[:dda]))

In [ ]:
save("saved-pulses/single_qubit_gateset_default.jld2", prob.trajectory)

In [ ]:
writedlm("saved-pulses/a_single_qubit_gateset_default.csv", prob.trajectory[:a], ",")

In [ ]:
plot_control(accumulate(+, timesteps(prob.trajectory)), [1 + π, 1, 1 + π, 1] .* prob.trajectory[:a])


In [ ]:
function random_a_guess(traj::NamedTrajectory)
    # Positive (symmetric) upper bounds
    a_bounds = traj.bounds[:a][2]

    a_dists = [Uniform(
        -(a_bounds[i] == Inf ? 1.0 : a_bounds[i]),
        (a_bounds[i] == Inf ? 1.0 : a_bounds[i])
    ) for i = 1:traj.dims[:a]]

    a = hcat([
        zeros(traj.dims[:a]),
        vcat([rand(a_dists[i], 1, T - 2) for i = 1:traj.dims[:a]]...),
        zeros(traj.dims[:a])
    ]...)
    return a
end

# Offset initial controls with amplitude contraint

Note: The option 1, where we don't stagger, led to pulses that were not good. The initialization via the stagger is important?

In [ ]:
trajectory = copy(prob.trajectory)
# Need random initial conditions to avoid local minima

# Option 1:
# ----------------
# Don't stagger! Try to use the same order of magnitude as the initial gates.
# update!(trajectory, :a, [π/2/10 , π/2/10, π/4/10, π/4/10] .* random_a_guess(trajectory))
# update!(trajectory, :Ũ⃗, 2 * rand(trajectory.dims[:Ũ⃗], T) .- 1)

# Option 2:
# ----------------
# Stagger as in the plot above
update!(trajectory, :a, [1 + π, 1, 1 + π, 1] .* prob.trajectory[:a])
update!(trajectory, :Ũ⃗, 2 * rand(trajectory.dims[:Ũ⃗], T) .- 1)

parameters = deepcopy(prob.params)

objective = DefaultObjective()
objective += Objective(parameters[:objective_terms][1])
objective += QuadraticRegularizer(:dda, trajectory, 1e-3)
objective += QuadraticRegularizer(:a, trajectory, 1e-6)

update_bound!(trajectory, :a, 1.0)
constraints = trajectory_constraints(trajectory)

ipopt_options = Options()
ipopt_options.hessian_approximation = "limited-memory"
ipopt_options.recalc_y = "yes"
ipopt_options.recalc_y_feas_tol = 1e-2

rob_prob_1 = UnitaryRobustnessProblem(
    H_crosstalk,
    trajectory,
    prob.system,
    objective,
    prob.integrators,
    constraints;
    final_fidelity=0.9, 
    verbose=false,
    build_trajectory_constraints=false,
    hessian_approximation=true,
    ipopt_options=ipopt_options
)

In [ ]:
solve!(rob_prob_1; max_iter=100)

println("Fidelity: ", unitary_fidelity(rob_prob_1))
println("Robustness: ", infidelity_robustness(H_crosstalk, rob_prob_1))

In [ ]:
plot_control(rob_prob_1)

In [ ]:
trajectory = copy(rob_prob_1.trajectory)
parameters = deepcopy(prob.params)

objective = DefaultObjective()
objective += QuadraticRegularizer(:dda, trajectory, 1e-5)
objective += QuadraticRegularizer(:a, trajectory, 1e-5)

update_bound!(trajectory, :a, Inf)
update_bound!(trajectory, :dda, Inf)
constraints = trajectory_constraints(trajectory)

ipopt_options = Options()
ipopt_options.hessian_approximation = "limited-memory"
ipopt_options.recalc_y = "yes"
ipopt_options.recalc_y_feas_tol = 1e-2

rob_prob_2 = UnitaryRobustnessProblem(
    H_crosstalk,
    trajectory,
    prob.system,
    objective,
    prob.integrators,
    constraints;
    final_fidelity=0.9999,
    verbose=false,
    build_trajectory_constraints=false,
    hessian_approximation=true,
    ipopt_options=ipopt_options
)

In [ ]:
solve!(rob_prob_2; max_iter=200)

println("Fidelity: ", unitary_fidelity(rob_prob_2))
println("Robustness: ", infidelity_robustness(H_crosstalk, rob_prob_2))

In [ ]:
println("max(a): ", maximum(rob_prob_2.trajectory[:a]))
println("max(dda): ", maximum(rob_prob_2.trajectory[:dda]))

In [ ]:
plot_control(rob_prob_2)

In [ ]:
save("saved-pulses/single_qubit_gateset_R1e-5.jld2", rob_prob_2.trajectory)

In [ ]:
writedlm("saved-pulses/a_single_qubit_gateset_R1e-5.csv", rob_prob_2.trajectory[:a], ",")

# From 1.0 bound gate, no constraint, tighten R's

In [ ]:
trajectory = copy(rob_prob_1.trajectory)
# Need random initial conditions to avoid local minima
# update!(trajectory, :a, random_a_guess(trajectory))
# update!(trajectory, :Ũ⃗, 2 * rand(trajectory.dims[:Ũ⃗], T) .- 1)
parameters = deepcopy(prob.params)

objective = DefaultObjective()
objective += QuadraticRegularizer(:dda, trajectory, 1e-3)
objective += QuadraticRegularizer(:a, trajectory, 1e-3)

update_bound!(trajectory, :a, Inf)
update_bound!(trajectory, :dda, Inf)
constraints = trajectory_constraints(trajectory)

ipopt_options = Options()
ipopt_options.hessian_approximation = "limited-memory"
ipopt_options.recalc_y = "yes"
ipopt_options.recalc_y_feas_tol = 1e-3

rob_prob_3 = UnitaryRobustnessProblem(
    H_crosstalk,
    trajectory,
    prob.system,
    objective,
    prob.integrators,
    constraints;
    final_fidelity=0.9999,
    verbose=false,
    build_trajectory_constraints=false,
    hessian_approximation=true,
    ipopt_options=ipopt_options
)

In [ ]:
solve!(rob_prob_3; max_iter=50)

println("Fidelity: ", unitary_fidelity(rob_prob_3))
println("Robustness: ", infidelity_robustness(H_crosstalk, rob_prob_3))

In [ ]:
println("max(a): ", maximum(rob_prob_3.trajectory[:a]))
println("max(dda): ", maximum(rob_prob_3.trajectory[:dda]))

In [ ]:
plot_control(rob_prob_3)

In [ ]:
save("saved-pulses/single_qubit_gateset_R1e-3.jld2", rob_prob_3.trajectory)

In [ ]:
writedlm("saved-pulses/a_single_qubit_gateset_R1e-3.csv", rob_prob_3.trajectory[:a], ",")

# From 1.0 bound gate, no constraint, loosen R

In [ ]:
trajectory = copy(rob_prob_1.trajectory)
parameters = deepcopy(prob.params)

objective = DefaultObjective()
objective += QuadraticRegularizer(:dda, trajectory, 1e-6)
objective += QuadraticRegularizer(:a, trajectory, 1e-6)

update_bound!(trajectory, :a, Inf)
update_bound!(trajectory, :dda, Inf)
constraints = trajectory_constraints(trajectory)

ipopt_options = Options()
ipopt_options.hessian_approximation = "limited-memory"
ipopt_options.recalc_y = "yes"
ipopt_options.recalc_y_feas_tol = 1e-2

rob_prob_4 = UnitaryRobustnessProblem(
    H_crosstalk,
    trajectory,
    prob.system,
    objective,
    prob.integrators,
    constraints;
    final_fidelity=0.9999,
    verbose=false,
    build_trajectory_constraints=false,
    hessian_approximation=true,
    ipopt_options=ipopt_options
)

In [ ]:
solve!(rob_prob_4; max_iter=100)

println("Fidelity: ", unitary_fidelity(rob_prob_4))
println("Robustness: ", infidelity_robustness(H_crosstalk, rob_prob_4))

In [ ]:
println("max(a): ", maximum(rob_prob_4.trajectory[:a]))
println("max(dda): ", maximum(rob_prob_4.trajectory[:dda]))

In [ ]:
plot_control(rob_prob_4)

In [ ]:
save("saved-pulses/single_qubit_gateset_R1e-6.jld2", rob_prob_4.trajectory)

In [ ]:
writedlm("saved-pulses/a_single_qubit_gateset_R1e-6.csv", rob_prob_4.trajectory[:a], ",")

# From 1.0 bound gate, no constraint, tightest constraint

In [ ]:
trajectory = copy(rob_prob_1.trajectory)
parameters = deepcopy(prob.params)

objective = DefaultObjective()
objective += QuadraticRegularizer(:dda, trajectory, 1e-2)
objective += QuadraticRegularizer(:a, trajectory, 1e-2)

update_bound!(trajectory, :a, Inf)
update_bound!(trajectory, :dda, Inf)
constraints = trajectory_constraints(trajectory)

ipopt_options = Options()
ipopt_options.hessian_approximation = "limited-memory"
ipopt_options.recalc_y = "yes"
ipopt_options.recalc_y_feas_tol = 1e-2

rob_prob_5 = UnitaryRobustnessProblem(
    H_crosstalk,
    trajectory,
    prob.system,
    objective,
    prob.integrators,
    constraints;
    final_fidelity=0.9999,
    verbose=false,
    build_trajectory_constraints=false,
    hessian_approximation=true,
    ipopt_options=ipopt_options
)

In [ ]:
solve!(rob_prob_5; max_iter=10)

println("Fidelity: ", unitary_fidelity(rob_prob_5))
println("Robustness: ", infidelity_robustness(H_crosstalk, rob_prob_5))

In [ ]:
println("max(a): ", maximum(rob_prob_5.trajectory[:a]))
println("max(dda): ", maximum(rob_prob_5.trajectory[:dda]))

In [ ]:
plot_control(rob_prob_5)

In [ ]:
save("saved-pulses/single_qubit_gateset_R1e-2.jld2", rob_prob_5.trajectory)

# From 1.0 bound gate, no constraint, max regularization

In [ ]:
trajectory = copy(rob_prob_1.trajectory)
parameters = deepcopy(prob.params)

objective = DefaultObjective()
objective += QuadraticRegularizer(:dda, trajectory, 1e-1)
objective += QuadraticRegularizer(:a, trajectory, 1e-1)

update_bound!(trajectory, :a, Inf)
update_bound!(trajectory, :dda, Inf)
constraints = trajectory_constraints(trajectory)

ipopt_options = Options()
ipopt_options.hessian_approximation = "limited-memory"
ipopt_options.recalc_y = "yes"
ipopt_options.recalc_y_feas_tol = 1e-2

rob_prob_6 = UnitaryRobustnessProblem(
    H_crosstalk,
    trajectory,
    prob.system,
    objective,
    prob.integrators,
    constraints;
    final_fidelity=0.9999,
    verbose=false,
    build_trajectory_constraints=false,
    hessian_approximation=true,
    ipopt_options=ipopt_options
)

In [ ]:
solve!(rob_prob_6; max_iter=50)

println("Fidelity: ", unitary_fidelity(rob_prob_6))
println("Robustness: ", infidelity_robustness(H_crosstalk, rob_prob_6))

In [ ]:
println("max(a): ", maximum(rob_prob_6.trajectory[:a]))
println("max(dda): ", maximum(rob_prob_6.trajectory[:dda]))

In [ ]:
plot_control(rob_prob_6)

In [ ]:
save("saved-pulses/single_qubit_gateset_R1e-1.jld2", rob_prob_6.trajectory)